<div style="background:linear-gradient(135deg,#1a0d0d,#3b1a13,#5f2a0f);padding:44px 38px;border-radius:16px;color:#f6f0f0;font-family:'Segoe UI',sans-serif;border:1px solid #3d3030;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Data Provenance · Leakage Audit · Publication Prerequisite</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f6f0f0 !important;">Veri Kümesi Bütünlük Denetimi</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f6f0f0 !important;">Kullanılan &ldquo;balanced&rdquo; sürümdeki NORMAL sınıfı nereden geliyor ve bölmeler arasında hasta sızıntısı var mı?</h2>
  <hr style="border:0;border-top:1px solid #3d3030;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Donanım:</b> GPU gerekmez (CPU yeterli)</div>
    <div><b>Süre:</b> ~6 dakika</div>
    <div><b>Girdi:</b> yalnızca sınıflandırma veri kümesi</div>
    <div><b>Çıktı:</b> sızıntı ve köken raporu</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(253,120,56,.12);border-left:4px solid #fd7838;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Neden zorunlu:</b> Kullanılan kümede PNEUMONIA sayısı Kermany özgün sürümüyle neredeyse birebir aynı (4.265 ≈ 4.273), ancak NORMAL sınıfı <b>2,69 kat</b> büyük (4.265 ≈ 1.583 yerine) — yani <b>2.682 görüntü fazladan eklenmiş.</b> İki olasılık vardır ve ikisi de makaleyi doğrudan ilgilendirir: (i) çoğaltma/artırma kopyaları eklenmişse aynı hasta hem eğitimde hem testte bulunabilir ve iç test AUC'si (0,998) şişkindir; (ii) başka bir kaynaktan görüntü eklenmişse &ldquo;NORMAL vs PNEUMONIA&rdquo; kısmen &ldquo;kaynak A vs kaynak B&rdquo; hâline gelir — bu, A kolunun akciğere hiç bakmadan (LFR 0,005) %99,8 AUC üretmesini de açıklar. Hakem bu soruyu mutlaka soracaktır; yanıtı yayından önce bilinmelidir.
  </div>
</div>

## Ne denetleniyor

| # | Denetim | Neyi ortaya çıkarır |
|---|---|---|
| 1 | **Dosya adı şeması** | Kermany şemasına uymayan adlar → yabancı kaynak |
| 2 | **Hasta kimliği çıkarımı** | `person123_*` ve `IM-####-*` kalıplarından hasta anahtarı |
| 3 | **Hasta düzeyi sızıntı** | Aynı hasta hem eğitimde hem doğrulama/testte mi? |
| 4 | **Birebir kopya** | İçerik MD5'i aynı olan dosyalar |
| 5 | **Yakın kopya** | Algısal karma (dHash) Hamming ≤ 3 → çoğaltma/artırma izi |
| 6 | **Boyut dağılımı** | Sınıfa göre çözünürlük kümelenmesi → kaynak farkı işareti |

Bölme, ablasyon koşumundaki kodun **birebir aynısıyla** yeniden üretilir; imzalar
denetlenir.

**Girdi:** yalnızca `yusufmurtaza01/chest-xray-pneumonia-balanced-dataset`.
Accelerator **None** seçebilirsiniz — GPU gerekmez.

In [ ]:
import os, re, json, hashlib, random, warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

SEED = 42
NEAR_DUP_MAX_HAMMING = 3
WORK = "/kaggle/working"

plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white",
                     "savefig.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8A9499", "grid.color": "#D8DFE1",
                     "legend.frameon": False})

INPUT = "/kaggle/input"
def find_classification_root(root=INPUT):
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0] if hits else None

BASE = find_classification_root()
assert BASE is not None, "Veri kumesi bulunamadi."
print("Veri kumesi:", BASE)

# --- Tum dosyalari topla ---
files = []
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        d = os.path.join(BASE, split, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                files.append({"split": split, "sinif": cls, "ad": f,
                              "yol": os.path.join(d, f)})
df = pd.DataFrame(files)
print(f"\nToplam {len(df)} goruntu")
print(df.groupby(["split", "sinif"]).size().unstack(fill_value=0).to_string())

In [ ]:
# ── 1) Dosya adi semalari ────────────────────────────────────────────────
def scheme(name):
    b = os.path.splitext(name)[0]
    return re.sub(r"\d+", "#", b)

df["sema"] = df["ad"].map(scheme)
print("=" * 84)
print("  DOSYA ADI SEMALARI  (sinifa gore, ilk 12)")
print("=" * 84)
for cls in ["NORMAL", "PNEUMONIA"]:
    c = Counter(df[df.sinif == cls]["sema"])
    print(f"\n  {cls}  ({len(c)} farkli sema)")
    for s, n in c.most_common(12):
        print(f"    {n:>6}  {s}")

KERMANY = re.compile(r"^(person#_(bacteria|virus)_#|(NORMAL#-)?IM-#-#|IM-#-#-#)$")
df["kermany_semasi"] = df["sema"].map(lambda s: bool(KERMANY.match(s)))
print("\n" + "=" * 84)
print("  KERMANY SEMASINA UYMAYAN DOSYALAR")
print("=" * 84)
t = df.groupby(["sinif", "kermany_semasi"]).size().unstack(fill_value=0)
print(t.to_string())
yab = df[~df.kermany_semasi]
if len(yab):
    print(f"\n  {len(yab)} dosya Kermany semasina uymuyor -> YABANCI KAYNAK ISARETI")
    print("  Ornekler:", list(yab["ad"].head(8)))
else:
    print("\n  Tum dosyalar Kermany adlandirma semasina uyuyor.")

In [ ]:
# ── 2) Hasta kimligi + 3) hasta duzeyi sizinti ───────────────────────────
def patient_key(name):
    b = os.path.splitext(name)[0]
    m = re.match(r"(person\d+)_", b, re.I)
    if m:
        return "p_" + m.group(1).lower()
    m = re.match(r"(?:NORMAL\d*-)?IM-(\d+)", b, re.I)
    if m:
        return "im_" + m.group(1)
    return None

df["hasta"] = df["ad"].map(patient_key)
bilinmeyen = int(df["hasta"].isna().sum())
print(f"Hasta anahtari cikarilamayan: {bilinmeyen} / {len(df)}")
print(f"Farkli hasta sayisi: {df['hasta'].nunique()}")
print(df.groupby("sinif")["hasta"].nunique().to_string())

# Ablasyondaki bolmeyi birebir yeniden uret
def file_sig(p):
    return f"{os.path.basename(p)}_{os.path.getsize(p) if os.path.exists(p) else 0}"

random.seed(SEED)
tr = df[df.split == "train"]
pool = df[df.split.isin(["val", "test"])].copy()
tr_sigs = set(file_sig(p) for p in tr["yol"])
pool = pool[~pool["yol"].map(lambda p: file_sig(p) in tr_sigs)]
npool = pool[pool.sinif == "NORMAL"].to_dict("records")
ppool = pool[pool.sinif == "PNEUMONIA"].to_dict("records")
random.shuffle(npool); random.shuffle(ppool)
per = min(len(npool), len(ppool)) // 2
val_s  = npool[:per] + ppool[:per]
test_s = npool[per:2*per] + ppool[per:2*per]
random.shuffle(val_s); random.shuffle(test_s)

SPLITS = {"train": tr.to_dict("records"), "val": val_s, "test": test_s}
def fp(recs):
    return hashlib.md5("|".join(sorted(r["ad"] for r in recs)).encode()).hexdigest()[:16]
print("\nBolme imzalari:", {k: fp(v) for k, v in SPLITS.items()})
print("  (ablasyon: train ccf23597ec992137 · val 779ac7f6455a7ac8 · test 3a25cbb40ab471d7)")

hastalar = {k: set(r["hasta"] for r in v if r["hasta"]) for k, v in SPLITS.items()}
print("\n" + "=" * 84)
print("  HASTA DUZEYI SIZINTI DENETIMI")
print("=" * 84)
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ort = hastalar[a] & hastalar[b]
    n_img = sum(1 for r in SPLITS[b] if r["hasta"] in ort)
    print(f"  {a:<6} <-> {b:<5}: {len(ort):>5} ortak hasta | {b} tarafinda {n_img} goruntu etkileniyor")
print("=" * 84)
print("  NOT: Dosya adi + boyut denetimi bu tur sizintiyi YAKALAYAMAZ.")

In [ ]:
# ── 4) Birebir kopya (icerik MD5) + 5) yakin kopya (dHash) ───────────────
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def dhash64(path, size=8):
    im = Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return np.packbits(bits).view(np.uint64)[0]

print("Karma hesaplaniyor (~birkac dakika)...")
t0 = time.time()
md5s, dhs, sizes = [], [], []
for i, p in enumerate(df["yol"]):
    try:
        md5s.append(md5_of(p)); dhs.append(dhash64(p))
        with Image.open(p) as im:
            sizes.append(im.size)
    except Exception:
        md5s.append(None); dhs.append(np.uint64(0)); sizes.append((0, 0))
    if (i + 1) % 2000 == 0:
        print(f"  {i+1}/{len(df)} ({time.time()-t0:.0f} sn)")
df["md5"] = md5s
df["dhash"] = np.array(dhs, dtype=np.uint64)
df["w"] = [s[0] for s in sizes]; df["h"] = [s[1] for s in sizes]
print(f"tamam ({time.time()-t0:.0f} sn)")

# --- birebir kopya ---
dup = df[df.duplicated("md5", keep=False) & df["md5"].notna()]
print("\n" + "=" * 84)
print("  BIREBIR KOPYA (icerik MD5)")
print("=" * 84)
print(f"  Kopya iceren dosya sayisi: {len(dup)} | farkli kopya grubu: {dup['md5'].nunique()}")
if len(dup):
    g = dup.groupby("md5").agg(n=("ad", "size"), siniflar=("sinif", lambda s: sorted(set(s))),
                               splitler=("split", lambda s: sorted(set(s))))
    print(f"  Birden fazla SINIFTA gecen kopya grubu: {int(g['siniflar'].map(len).gt(1).sum())}")
    print(f"  Birden fazla SPLITTE gecen kopya grubu : {int(g['splitler'].map(len).gt(1).sum())}")
    print(g.head(10).to_string())

In [ ]:
# ── Yakin kopya: dHash Hamming mesafesi ──────────────────────────────────
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
H = df["dhash"].values.astype(np.uint64)
n = len(H)
print(f"Yakin kopya taramasi ({n} x {n})...")
pairs = []
CH = 400
t0 = time.time()
for s in range(0, n, CH):
    e = min(s + CH, n)
    x = np.bitwise_xor.outer(H[s:e], H)          # (c, n) uint64
    d = POP[x.view(np.uint8).reshape(x.shape[0], n, 8)].sum(-1)
    d[np.arange(e - s), np.arange(s, e)] = 99     # kendisi
    ii, jj = np.where(d <= NEAR_DUP_MAX_HAMMING)
    for a, b in zip(ii + s, jj):
        if a < b:
            pairs.append((int(a), int(b), int(d[a - s, b])))
    if (s // CH) % 5 == 0:
        print(f"  {e}/{n} ({time.time()-t0:.0f} sn)")
print(f"tamam ({time.time()-t0:.0f} sn) | {len(pairs)} yakin cift")

if pairs:
    pr = pd.DataFrame(pairs, columns=["i", "j", "mesafe"])
    for c in ["split", "sinif", "ad", "hasta"]:
        pr[c + "_i"] = df[c].values[pr["i"]]
        pr[c + "_j"] = df[c].values[pr["j"]]
    pr["farkli_split"] = pr["split_i"] != pr["split_j"]
    pr["farkli_sinif"] = pr["sinif_i"] != pr["sinif_j"]
    print("\n" + "=" * 84)
    print(f"  YAKIN KOPYA (dHash Hamming <= {NEAR_DUP_MAX_HAMMING})")
    print("=" * 84)
    print(f"  Toplam cift            : {len(pr)}")
    print(f"  Farkli SPLIT arasi     : {int(pr['farkli_split'].sum())}   <-- sizinti riski")
    print(f"  Farkli SINIF arasi     : {int(pr['farkli_sinif'].sum())}   <-- etiket celiskisi")
    print(f"  Etkilenen dosya sayisi : {len(set(pr['i']) | set(pr['j']))}")
    print("\n  Ornek ciftler:")
    print(pr[["ad_i", "split_i", "sinif_i", "ad_j", "split_j", "sinif_j", "mesafe"]].head(10).to_string(index=False))
    pr.to_csv(os.path.join(WORK, "audit_near_duplicates.csv"), index=False)
else:
    pr = pd.DataFrame()
    print("  Yakin kopya bulunamadi.")

In [ ]:
# ── 6) Boyut dagilimi: kaynak farki isareti ──────────────────────────────
df["en_boy"] = df["w"] / df["h"].replace(0, np.nan)
print("=" * 84)
print("  COZUNURLUK DAGILIMI (sinifa gore)")
print("=" * 84)
print(df.groupby("sinif")[["w", "h", "en_boy"]].describe().round(2).to_string())
print("\n  En sik gorulen (genislik x yukseklik) degerleri:")
for cls in ["NORMAL", "PNEUMONIA"]:
    c = Counter(zip(df[df.sinif == cls]["w"], df[df.sinif == cls]["h"]))
    top = ", ".join(f"{w}x{h} ({n})" for (w, h), n in c.most_common(5))
    print(f"    {cls:<10}: {top}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for cls, col in [("NORMAL", "#2E8B57"), ("PNEUMONIA", "#B04A1E")]:
    sub = df[df.sinif == cls]
    axes[0].scatter(sub["w"], sub["h"], s=5, alpha=.25, color=col, label=cls, edgecolor="none")
    axes[1].hist(sub["en_boy"].dropna(), bins=60, alpha=.55, color=col, label=cls, density=True)
axes[0].set_xlabel("genislik (px)"); axes[0].set_ylabel("yukseklik (px)")
axes[0].set_title("Cozunurluk dagilimi"); axes[0].legend(); axes[0].grid(alpha=.4)
axes[1].set_xlabel("en / boy orani"); axes[1].set_ylabel("yogunluk")
axes[1].set_title("En-boy orani"); axes[1].legend(); axes[1].grid(alpha=.4)
fig.suptitle("Sinifa gore goruntu geometrisi — ayrik kumelenme kaynak farkina isaret eder",
             fontsize=11.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "audit_fig_01_geometry.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Rapor ────────────────────────────────────────────────────────────────
KERMANY_REF = {"NORMAL": 1583, "PNEUMONIA": 4273}
say = df.groupby("sinif").size().to_dict()

ozet = {
    "toplam_goruntu": int(len(df)),
    "sinif_sayilari": {k: int(v) for k, v in say.items()},
    "kermany_referans": KERMANY_REF,
    "fazlalik": {k: int(say.get(k, 0) - KERMANY_REF[k]) for k in KERMANY_REF},
    "kermany_semasina_uymayan": int((~df.kermany_semasi).sum()),
    "hasta_anahtari_yok": int(df["hasta"].isna().sum()),
    "farkli_hasta": int(df["hasta"].nunique()),
    "birebir_kopya_dosya": int(len(dup)),
    "yakin_kopya_cift": int(len(pr)),
    "yakin_kopya_farkli_split": int(pr["farkli_split"].sum()) if len(pr) else 0,
    "yakin_kopya_farkli_sinif": int(pr["farkli_sinif"].sum()) if len(pr) else 0,
    "hasta_sizintisi": {f"{a}-{b}": len(hastalar[a] & hastalar[b])
                        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]},
}
with open(os.path.join(WORK, "audit_summary.json"), "w", encoding="utf-8") as f:
    json.dump(ozet, f, indent=2, ensure_ascii=False)
df.drop(columns=["yol"]).to_csv(os.path.join(WORK, "audit_files.csv"), index=False)

print("=" * 84)
print("  DENETIM RAPORU")
print("=" * 84)
for k in KERMANY_REF:
    print(f"  {k:<10}: {say.get(k,0):>5}  (Kermany: {KERMANY_REF[k]})  "
          f"fazlalik: {ozet['fazlalik'][k]:+d}")
print(f"\n  Kermany semasina uymayan dosya : {ozet['kermany_semasina_uymayan']}")
print(f"  Hasta anahtari cikarilamayan   : {ozet['hasta_anahtari_yok']}")
print(f"  Birebir kopya dosya            : {ozet['birebir_kopya_dosya']}")
print(f"  Yakin kopya cift               : {ozet['yakin_kopya_cift']}"
      f"  (farkli split: {ozet['yakin_kopya_farkli_split']}, "
      f"farkli sinif: {ozet['yakin_kopya_farkli_sinif']})")
print(f"  Hasta sizintisi                : {ozet['hasta_sizintisi']}")
print("=" * 84)

sorunlar = []
if ozet["kermany_semasina_uymayan"] > 0:
    sorunlar.append("Yabanci adlandirma semasi -> NORMAL sinifina baska kaynaktan goruntu eklenmis olabilir")
if ozet["hasta_sizintisi"]["train-test"] > 0 or ozet["hasta_sizintisi"]["val-test"] > 0:
    sorunlar.append("HASTA DUZEYI SIZINTI -> ic test metrikleri iyimser")
if ozet["yakin_kopya_farkli_split"] > 0:
    sorunlar.append("Bolmeler arasi yakin kopya -> cogaltma kaynakli sizinti")
if ozet["yakin_kopya_farkli_sinif"] > 0:
    sorunlar.append("Siniflar arasi yakin kopya -> etiket celiskisi")

print("\n  BULGULAR")
if sorunlar:
    for s in sorunlar:
        print(f"    - {s}")
    print("\n  Bu bulgular makalenin Kisitlar bolumunde ACIKCA belirtilmelidir.")
else:
    print("    - Belirgin bir butunluk sorunu saptanmadi.")
print("=" * 84)

## Sonucun makale açısından anlamı

**Yabancı adlandırma şeması bulunursa** → NORMAL sınıfına başka bir kaynaktan görüntü
eklenmiştir. Bu durumda "NORMAL vs PNEUMONIA" kısmen "kaynak A vs kaynak B" demektir ve
A kolunun akciğere hiç bakmadan %99,8 AUC üretmesi doğal karşılanır. Makalede veri kümesi
şu şekilde tanımlanmalıdır: *"Kermany ve ark. veri kümesinin, NORMAL sınıfı X kaynağından
tamamlanmış dengeli bir türevi"* — "Kermany veri kümesi" demek yanlış olur.

**Hasta düzeyi sızıntı bulunursa** → iç test AUC'si (0,998) iyimserdir. Dış doğrulama
sonuçları etkilenmez (bağımsız kümeler), dolayısıyla makalenin ana bulguları ayakta kalır;
ancak iç test sayısı "aynı hastaların farklı filmleri" uyarısıyla verilmelidir.

**Bölmeler arası yakın kopya bulunursa** → çoğaltma ile dengeleme yapılmıştır. Aynı
uyarı geçerlidir, ek olarak eğitim setinin efektif büyüklüğü raporlanandan küçüktür.

**Hiçbiri bulunmazsa** → veri kümesi temizdir ve bu denetim, makalenin metodoloji
bölümünde bir güç unsuru olarak raporlanabilir: *"Bölmeler hasta düzeyinde ve algısal
karma ile sızıntı açısından denetlenmiştir."*

> Her durumda bu notebook'un çıktısı makalenin **Veri** ve **Kısıtlar** bölümlerini
> yazmak için gereklidir. Sonuç ne olursa olsun raporlanması, hakem sürecinde
> savunulabilirliği artırır.